In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("source_table", "oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref", "Source Table")
dbutils.widgets.text("destination_table", "oh_apm_stg.apm_report.Load_Report_Log_CMC_CPC_Attr", "Destination Table")

In [0]:

source_table = dbutils.widgets.get("source_table")
destination_table = dbutils.widgets.get("destination_table")
print(f"📌 Source Table: {source_table}")
print(f"📌 Destination Table: {destination_table}")

In [0]:
try:
    # Get the flag from Prevalidation
    run_flag = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="run_flag", debugValue=False)
    print(f"✅ Retrieved run_flag: {run_flag}")

    if run_flag:
        print("🚀 Flag is True. Proceeding with data copy...")

        # Get dates from DateProvider
        start_date = dbutils.jobs.taskValues.get(taskKey="End_Date_time", key="start_date", debugValue="1900-01-01")
        end_date = dbutils.jobs.taskValues.get(taskKey="End_Date_time", key="end_date", debugValue="1900-01-01")

        print(f"📌 Start Date: {start_date}")
        print(f"📌 End Date: {end_date}")

        # Run the SQL with WHERE clause to filter by dates
        spark.sql(f"""
            INSERT INTO {destination_table}
            SELECT
                File_Name,
                Date_Received_by_APM,
                Status AS Load_Status,
                Number_of_Records_Received,
                Number_of_Records_Loaded,
                Number_of_Error_Records,
                TO_DATE(Start_Load_Date, 'yyyy-MM-dd HH:mm:ss') AS Start_Load_Date,
                TO_DATE(End_Load_Date, 'yyyy-MM-dd HH:mm:ss') AS End_Load_Date
            FROM {source_table}
            WHERE Start_Load_Date = '{start_date}' AND End_Load_Date = '{end_date}'
        """)

        print("✅ Data successfully copied to destination table.")
    else:
        print("⚠️ Flag is False. Skipping data copy.")

except Exception as e:
    print("❌ An error occurred during the CopyToAPMTables task.")
    print(f"Error details: {str(e)}")
    raise
